# Spot RL Training -- Kaggle
Train DQN generalist agent for AWS Spot Instance optimization.

In [ ]:
# Step 0: Tham số của lần chạy.
# Workflow .github/workflows/train-submit.yml thay 2 placeholder bên dưới
# trước khi `kaggle kernels push`. Chạy tay thì để nguyên → dùng mặc định.
GIT_SHA = '__GIT_SHA__'
RUN_CONFIG = '__RUN_CONFIG__'

GIT_SHA = None if GIT_SHA.startswith('__') else GIT_SHA
RUN_CONFIG = None if RUN_CONFIG.startswith('__') else RUN_CONFIG
print('GIT_SHA   :', GIT_SHA or '(mới nhất)')
print('RUN_CONFIG:', RUN_CONFIG or '(mặc định ở Step 5)')

In [ ]:
# Step 1: Install dependencies
!pip install gymnasium torch numpy pandas pyarrow pyyaml mlflow dagshub --quiet

In [ ]:
# ── RunPod Setup (thay thế cell-2 khi chạy trên RunPod) ──────────────────────
# Chạy cell này THAY VÌ cell-2 khi dùng RunPod Jupyter.
# Kaggle: dùng cell-2 (auto-detect /kaggle/input), bỏ qua cell này.
import os
if os.path.exists('/kaggle/working'):
    print('Kaggle → bỏ qua cell RunPod setup')
else:
    import os, sys, subprocess

    REPO_URL = 'https://github.com/RibunLoc/NT549.Q21_spot-rl-optimizer'
    DST = '/workspace/spot-rl'

    if not os.path.exists(DST):
        subprocess.run(['git', 'clone', '--depth=1', REPO_URL, DST], check=True)
        print(f'Cloned to {DST}')
    else:
        subprocess.run(['git', '-C', DST, 'pull', '--ff-only'], check=True)
        print(f'Updated {DST}')

    # Add to path
    if DST not in sys.path:
        sys.path.insert(0, DST)

    # Ensure __init__.py exists
    for pkg in ['envs', 'agents', 'utils']:
        init = os.path.join(DST, pkg, '__init__.py')
        if not os.path.exists(init):
            open(init, 'w').close()

    # Link data folder vào working dir
    os.makedirs('/workspace/working/models', exist_ok=True)
    if not os.path.exists('/workspace/working/data'):
        os.symlink(os.path.join(DST, 'data'), '/workspace/working/data')

    # Patch SAVE_DIR và sys.path cho các cell sau
    SAVE_DIR = '/workspace/working/models'
    os.chdir(DST)

    print('RunPod setup complete')
    print(f'  Repo : {DST}')
    print(f'  Models: {SAVE_DIR}')

In [ ]:
# Step 2: Lấy code từ GitHub, ĐÚNG commit mà workflow train-submit gửi sang.
# Chạy tay (GIT_SHA = None) → lấy commit mới nhất của nhánh mặc định.
import os, sys, subprocess

REPO_URL = 'https://github.com/RibunLoc/NT549.Q21_spot-rl-optimizer'
DST = '/kaggle/working/spot-rl'

def git(*args):
    return subprocess.run(['git', '-C', DST, *args], check=True,
                          capture_output=True, text=True).stdout.strip()

if not os.path.exists(DST):
    subprocess.run(['git', 'clone', '--depth=1', REPO_URL, DST], check=True)
# GitHub cho fetch thẳng theo SHA → không cần clone full history
git('fetch', '--depth=1', 'origin', GIT_SHA or 'HEAD')
git('checkout', '--detach', 'FETCH_HEAD')
GIT_SHA = git('rev-parse', 'HEAD')   # luôn ghi SHA thật, kể cả khi chạy tay
print(f'Code @ {GIT_SHA}')

# Ensure __init__.py for packages (some envs strip them when packaging)
for pkg in ['envs', 'agents', 'utils']:
    init = os.path.join(DST, pkg, '__init__.py')
    if not os.path.exists(init):
        open(init, 'w').close()

# Make `from envs import ...` work from any cell
if DST not in sys.path:
    sys.path.insert(0, DST)
os.chdir(DST)

print(f'Project root: {DST}')
print(f'CWD: {os.getcwd()}')
for folder in ['envs', 'agents', 'utils', 'data', '.dvc']:
    p = os.path.join(DST, folder)
    print(f'  {folder}/: {"OK" if os.path.exists(p) else "MISSING"}')


In [ ]:
# Step 2b: Pull data from DVC remote (S3)
# Requires Kaggle Secrets: AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY
# Optional: AWS_SESSION_TOKEN, AWS_DEFAULT_REGION
import os, subprocess, sys

def _set_aws_from_secrets():
    try:
        from kaggle_secrets import UserSecretsClient
        s = UserSecretsClient()
        os.environ['AWS_ACCESS_KEY_ID'] = s.get_secret('AWS_ACCESS_KEY_ID')
        os.environ['AWS_SECRET_ACCESS_KEY'] = s.get_secret('AWS_SECRET_ACCESS_KEY')
        try:
            os.environ['AWS_SESSION_TOKEN'] = s.get_secret('AWS_SESSION_TOKEN')
        except Exception:
            pass
        try:
            region = s.get_secret('AWS_DEFAULT_REGION')
        except Exception:
            region = 'ap-southeast-1'
        os.environ['AWS_DEFAULT_REGION'] = region or 'ap-southeast-1'
        os.environ['AWS_REGION'] = os.environ['AWS_DEFAULT_REGION']
        print('AWS credentials loaded from Kaggle Secrets')
    except Exception as e:
        print(f'Kaggle Secrets unavailable: {e}')

# Set credentials if missing. Values are never printed.
if not os.environ.get('AWS_ACCESS_KEY_ID'):
    _set_aws_from_secrets()

print('AWS env status:', {
    'AWS_ACCESS_KEY_ID': bool(os.environ.get('AWS_ACCESS_KEY_ID')),
    'AWS_SECRET_ACCESS_KEY': bool(os.environ.get('AWS_SECRET_ACCESS_KEY')),
    'AWS_SESSION_TOKEN': bool(os.environ.get('AWS_SESSION_TOKEN')),
    'AWS_DEFAULT_REGION': os.environ.get('AWS_DEFAULT_REGION'),
})

if not os.path.exists(os.path.join(DST, '.dvc', 'config')):
    raise FileNotFoundError(
        'Missing .dvc/config in /kaggle/working. Re-run Step 2 after updating the notebook, '
        'or rebuild/upload the Kaggle dataset so it includes the .dvc directory.'
    )

processed_ok = all(
    os.path.exists(os.path.join(DST, 'data', 'processed', f))
    for f in ['multipool_stable.csv', 'multipool_volatile.csv',
              'multipool_spike.csv', 'multipool_az_divergence.csv']
)

if processed_ok:
    print('Data already present - skipping dvc pull')
else:
    print('Pulling data from DVC remote (S3)...')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q', 'dvc[s3]'
    ])
    result = subprocess.run(
        [sys.executable, '-m', 'dvc', 'pull', 'data/processed.dvc', '-v'],
        cwd=DST, capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print('STDERR:', result.stderr)
        raise RuntimeError(
            'dvc pull failed. Check: Kaggle Internet is ON, AWS Secrets are enabled for this notebook, '
            'the IAM user can s3:GetObject/ListBucket on the DVC bucket, and .dvc/config exists.'
        )

# Verify
for fname in ['multipool_stable.csv', 'multipool_volatile.csv',
              'multipool_spike.csv', 'multipool_az_divergence.csv']:
    path = os.path.join(DST, 'data', 'processed', fname)
    size = os.path.getsize(path) // 1024 if os.path.exists(path) else 0
    status = f'{size} KB' if size else 'MISSING'
    print(f'  {fname}: {status}')


In [ ]:
# Step 3: Verify imports
import numpy as np
import torch
from experiments.common import load_config
from experiments.train_generalist import train
print(f'PyTorch: {torch.__version__} | Device: {"cuda" if torch.cuda.is_available() else "cpu"}')

In [ ]:
# Step 5: Chọn config. Mọi tham số env / agent / training nằm trong file YAML,
# dùng CHUNG với experiments/evaluate_kaggle.py --config (eval khớp đúng lúc train).
#   v41.yaml                    – tái hiện v41 (giữ nguyên lỗi LR + done) để làm mốc
#   s1_fix_lr_done.yaml         – bước 1: LR không đổi + done = terminated
#   s3_interrupt_realistic.yaml – bước 3: interrupt theo từng instance, ~10%/tháng
#   s3_interrupt_stress.yaml    – bước 3: stress, ~1 lần/instance/ngày
#   s4_pool_obs.yaml            – bước 4: feature từng pool + head op×pool (obs 254 dims)
#   s5_simple_reward.yaml       – bước 5: reward = -(cost + λ·job fail + phí đổi fleet)
# Mỗi file kế thừa file trước (extends), nên chạy lần lượt sẽ thấy từng thay đổi tác động bao nhiêu.
import os

DEFAULT_CONFIG = 'experiments/configs/generalist/s1_fix_lr_done.yaml'   # dùng khi chạy tay
CONFIG = RUN_CONFIG or DEFAULT_CONFIG
cfg = load_config(CONFIG)
SAVE_DIR = os.path.join('/kaggle/working/models', cfg['version'])
os.makedirs(SAVE_DIR, exist_ok=True)

print('config  :', CONFIG)
print('version :', cfg['version'])
print('sim     :', cfg['env']['sim'])
print('reward  :', cfg['env']['reward'].get('mode', 'shaped'))
print('agent   :', {k: cfg['agent'][k] for k in ['network_type', 'hidden_dim', 'learning_rate', 'lr_schedule']})
print('training: bootstrap_on_truncation =', cfg['training']['bootstrap_on_truncation'],
      '| episodes =', cfg['training']['num_episodes'])
print('SAVE_DIR:', SAVE_DIR)

# Resume (tùy chọn)
RESUME_CHECKPOINT = None   # vd: os.path.join(SAVE_DIR, 'checkpoint_ep3000.pth')
START_EPISODE = 0
RESUME_LR = None

In [ ]:
# ── RunPod MLflow setup (thay thế Step 5b khi chạy trên RunPod) ─────────────
# Set DAGSHUB_USER_TOKEN trong RunPod → Edit Pod → Environment Variables trước khi start pod.
# Kaggle: dùng Step 5b (Kaggle Secrets), bỏ qua cell này.
import os
if os.path.exists('/kaggle/working'):
    print('Kaggle → bỏ qua cell RunPod MLflow')
else:
    import mlflow

    token = os.environ.get('DAGSHUB_USER_TOKEN', '')
    if token:
        os.environ['MLFLOW_TRACKING_USERNAME'] = 'RibunLoc'
        os.environ['MLFLOW_TRACKING_PASSWORD'] = token
        MLFLOW_URI = 'https://dagshub.com/RibunLoc/NT549.Q21_spot-rl-optimizer.mlflow'
        mlflow.set_tracking_uri(MLFLOW_URI)
        mlflow.set_experiment('spot-rl-generalist')
        mlflow.enable_system_metrics_logging()
        try:
            mlflow.tracking.MlflowClient().search_experiments(max_results=1)
            print('MLflow (DagsHub):', MLFLOW_URI)
        except Exception as e:
            print(f'MLflow connection failed: {e}')
            MLFLOW_URI = None
    else:
        print('DAGSHUB_USER_TOKEN chưa set — tracking disabled')
        MLFLOW_URI = None

In [ ]:
# Step 5b: MLflow trên DagsHub — token lấy từ Kaggle Secret DAGSHUB_USER_TOKEN
# (DagsHub → User Settings → Tokens). Artifact/model lưu trên storage của DagsHub.
import os, mlflow

DAGSHUB_OWNER = 'RibunLoc'
DAGSHUB_REPO = 'NT549.Q21_spot-rl-optimizer'
MLFLOW_URI = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('DAGSHUB_USER_TOKEN')
    # dagshub.init đọc token từ env. Thiếu token nó sẽ mở OAuth trên trình duyệt
    # → treo khi notebook chạy headless (kaggle kernels push).
    os.environ['DAGSHUB_USER_TOKEN'] = token
    # MLflow client dùng basic auth: user = owner, password = token
    os.environ['MLFLOW_TRACKING_USERNAME'] = DAGSHUB_OWNER
    os.environ['MLFLOW_TRACKING_PASSWORD'] = token

    import dagshub
    dagshub.init(repo_owner=DAGSHUB_OWNER, repo_name=DAGSHUB_REPO, mlflow=True)
    MLFLOW_URI = mlflow.get_tracking_uri()
    mlflow.set_experiment('spot-rl-generalist')
    mlflow.enable_system_metrics_logging()
    mlflow.tracking.MlflowClient().search_experiments(max_results=1)
    print('MLflow (DagsHub):', MLFLOW_URI)
    print('System metrics logging: ENABLED (CPU/RAM/GPU)')
except Exception as e:
    print(f'MLflow unavailable: {e}')
    MLFLOW_URI = None

# Chạy từ CI mà không log được thì DỪNG: train không log = phí GPU, eval gate không thấy run.
if MLFLOW_URI is None and RUN_CONFIG:
    raise RuntimeError('Không kết nối được MLflow trên DagsHub. Kiểm tra Kaggle Secret DAGSHUB_USER_TOKEN.')

In [ ]:
# Step 6: Training — vòng lặp nằm trong experiments/train_generalist.py
# (eval-gated save, early stop, collapse guard, checkpoint, MLflow giống bản cũ).
if 'MLFLOW_URI' not in dir():
    MLFLOW_URI = None
RUN_TAGS = {
    'git_sha': GIT_SHA,
    'run_config': CONFIG,
    'trigger': 'github-actions' if RUN_CONFIG else 'manual',
}
history = train(
    cfg, save_dir=SAVE_DIR, mlflow_uri=MLFLOW_URI,
    resume_checkpoint=RESUME_CHECKPOINT, start_episode=START_EPISODE, resume_lr=RESUME_LR,
    platform='kaggle', tags=RUN_TAGS,
)

# Đánh dấu cho workflow train-check: run này đang chờ EVAL GATE.
# Không có model qua eval lúc train (best_eval.pth) thì đánh 'no_model' để gate bỏ qua.
if MLFLOW_URI and history.get('mlflow_run_id'):
    import mlflow
    gate = 'pending' if history.get('best_cost') is not None else 'no_model'
    mlflow.tracking.MlflowClient().set_tag(history['mlflow_run_id'], 'gate_status', gate)
    print(f"MLflow run {history['mlflow_run_id']} → gate_status={gate}")

In [ ]:
# Step 7: Plot training curves
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

def smooth(data, w=100):
    return pd.Series(data).rolling(w, min_periods=1).mean().values

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
fig.suptitle(f"{cfg['version']} -- DQN generalist on 4 scenarios", fontsize=13)
eps = range(1, len(history['rewards']) + 1)
for ax, key, color, title in [(axes[0, 0], 'rewards', 'steelblue', 'Reward'),
                              (axes[0, 1], 'cost', 'coral', 'Cost ($)'),
                              (axes[0, 2], 'sla', 'green', 'SLA')]:
    ax.plot(eps, history[key], alpha=0.15, color=color)
    ax.plot(eps, smooth(history[key]), color=color, lw=2)
    ax.set_title(f'{title} -- all scenarios'); ax.grid(alpha=0.3)
axes[0, 2].axhline(0.95, color='red', linestyle='--', alpha=0.5)

colors = {'stable': '#2ca02c', 'volatile': '#d62728', 'spike': '#ff7f0e', 'az_divergence': '#9467bd'}
for scn, col in colors.items():
    if len(history['per_scn_cost'].get(scn, [])) > 10:
        xs = np.arange(len(history['per_scn_cost'][scn]))
        axes[1, 0].plot(xs, smooth(history['per_scn_cost'][scn], 50), color=col, lw=2, label=scn)
        axes[1, 1].plot(xs, smooth(history['per_scn_sla'][scn], 50), color=col, lw=2, label=scn)
axes[1, 0].set_title('Cost per scenario'); axes[1, 0].grid(alpha=0.3); axes[1, 0].legend(fontsize=8)
axes[1, 1].set_title('SLA per scenario'); axes[1, 1].grid(alpha=0.3); axes[1, 1].legend(fontsize=8)
axes[1, 1].axhline(0.95, color='red', linestyle='--', alpha=0.5)

axes[1, 2].plot(eps, history['epsilon'], color='orange', lw=2, label='epsilon')
axes[1, 2].set_title('Epsilon & LR'); axes[1, 2].grid(alpha=0.3)
ax_lr = axes[1, 2].twinx()
ax_lr.plot(eps, history['lr'], color='purple', lw=1.5, label='lr')
ax_lr.set_yscale('log')
axes[1, 2].legend(loc='upper right', fontsize=8); ax_lr.legend(loc='lower left', fontsize=8)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/training_curves.png', dpi=100)
plt.show()

print('Per-scenario (last 100 eps):')
for scn in colors:
    if history['per_scn_cost'].get(scn):
        print(f"  {scn:15s}: cost=${np.mean(history['per_scn_cost'][scn][-100:]):6.2f}  "
              f"SLA={np.mean(history['per_scn_sla'][scn][-100:]):.1%}")

In [ ]:
# Step 8: List output files
import os
print('Files saved:')
for f in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(f'{SAVE_DIR}/{f}') / 1024
    print(f'  {f}: {size:.0f} KB')
